# 02 · Silver · `leituras`
### V-Energy Grid 360 — Arquitetura Medalhão

Lê `v_energy.bronze.leituras` (5 arquivos CSV, um por ano, 2022 a 2026; tudo como texto) e grava `v_energy.silver.leituras` com os dados limpos, tipados e confiáveis para a Gold.

**Fonte das regras:** `exploracao_leituras.ipynb` e a seção `leituras` do `docs/data_quality_log.md`.

| # | Tratamento | Linhas afetadas | Regra |
|---|---|---|---|
| 1 | Cópias exatas de linhas | 21.523 | `dropDuplicates(["id_leitura"])` |
| 2 | Linhas órfãs (`id_medidor` inexistente em `medidores`) | 8.609 | remover |
| 3 | `leitura_anterior` ausente em 4 formas (nulo real, `N/A`, `NULL`, `-1`) | 32.285 | nulo real + flag `leitura_anterior_ausente`; consumo mantido |
| 4 | Consumo negativo por sinal invertido | 1.717 | `abs(consumo)` + flag `consumo_corrigido` |
| 5 | Consumo negativo por troca de medidor | 6.299 | `consumo = leitura_atual` + flag `troca_medidor` |
| 6 | `data_leitura` em 4 formatos | ~258 mil fora do ISO | converter cada padrão com o seu formato e gravar como `date` |
| 7 | Consumo difere de `atual − anterior` em ±1 kWh | ~24% | **não corrigir** (arredondamento da origem); ±1 como tolerância nas validações |

**Ordem das operações (importa):**
1. Remover cópias e, depois, órfãs.
2. Marcar pelo **texto original**, antes de qualquer cast: o `.0` do consumo e as formas de ausência de `leitura_anterior`.
3. Converter os tipos e só então calcular a troca de medidor (`lag` por UC). Calcular a troca antes de remover as órfãs cria trocas falsas (23.314 em vez de 6.299).

**Decisões:** tomadas pela responsável pela tabela, com a autonomia combinada com o grupo para as tabelas de cada integrante, e registradas em `docs/data_quality_log.md` (itens L1 a L7). As alternativas consideradas e o motivo de cada escolha estão no log.

| Decisão | Regra no notebook | Alternativa considerada | Onde mudar, se for reaberta |
|---|---|---|---|
| Troca de medidor (L5) | `consumo_kwh = leitura_atual` + flag `troca_medidor` | consumo nulo + flag | primeiro `when` da célula da seção 9 |
| Órfãs (L2) | remover | quarentena (tabela própria) | gravar `df_orfas` em tabela (seção 5) |
| Sinal invertido (L4) | `abs(consumo)` + flag `consumo_corrigido` | consumo nulo; recalcular `atual − anterior` | segundo `when` da célula da seção 9 |
| `leitura_anterior` ausente (L3) | nulo + flag, sem recuperar | coluna `leitura_anterior_recuperada` (`lag` da `leitura_atual` por UC e medidor; recupera 97,6%) | acrescentar o `lag` na célula da seção 9 |

**Resultado esperado:** 2.152.336 linhas, com `(id_uc, mes_referencia)` único.

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from pyspark.sql import functions as F, Row, Window
from datetime import datetime

tabela_bronze = f"{bronze_schema}.leituras"
tabela_silver = f"{silver_schema}.leituras"

print(f"origem:  {tabela_bronze}")
print(f"destino: {tabela_silver}")

origem:  v_energy.bronze.leituras
destino: v_energy.silver.leituras


## 2. Funções de checagem de qualidade

Mesmas funções da aula `02_Bronze_to_Silver`, para que as três Silvers do grupo gravem em `silver.dq_log` com as mesmas colunas (`table_name`, `check_name`, `total_rows`, `failed_rows`, `passed`, `checked_at`).

- `dq_check(table_name, check_name, df, condition)`: `condition` é a condição **esperada** (o que deveria ser verdade em todas as linhas). Falham as linhas que a violam.
- `dq_check_unique(table_name, check_name, df, key_cols)`: passa se a chave não se repete.

As funções só acumulam os resultados na lista `dq_results`. Depois das checagens, o notebook faz duas coisas, nesta ordem: **grava o resultado em `silver.dq_log`** (sempre, inclusive quando há falha, porque o registro da falha também é útil) e **só então decide se grava a Silver** (só se todas as checagens passarem).

**Cuidado com nulos:** linhas em que a condição resulta em nulo não contam como falha (o filtro `~condition` descarta nulos). Quando o nulo também for um problema, escreva um `dq_check` separado com `isNotNull()`.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

## 3. Leitura da Bronze

Leitura da tabela Bronze, sem transformação nenhuma. Duas conferências antes de começar:
- A contagem deve ser **2.182.468** linhas (mesmo número da exploração).
- Todas as colunas devem aparecer como `string`: é a regra da Bronze. As conversões de tipo só acontecem aqui na Silver, depois que as marcas do texto original forem registradas.

O DataFrame `bronze` fica guardado para a comparação Bronze × Silver no fim do notebook.

In [0]:
bronze = spark.table(tabela_bronze)

n_bronze = bronze.count()
print(f"Linhas na Bronze: {n_bronze:,}".replace(",", "."))

bronze.printSchema()
display(bronze.limit(5))

Linhas na Bronze: 2.182.468
root
 |-- id_leitura: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- id_medidor: string (nullable = true)
 |-- data_leitura: string (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- leitura_anterior: string (nullable = true)
 |-- leitura_atual: string (nullable = true)
 |-- consumo_kwh: string (nullable = true)
 |-- tipo_leitura: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



id_leitura,id_uc,id_medidor,data_leitura,mes_referencia,leitura_anterior,leitura_atual,consumo_kwh,tipo_leitura,source_file,ingestion_timestamp
LT000000030,UC100000,MD500000,2025-01-18,2025-01,16946,17373,426,REAL,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv,2026-10-19T20:11:27.087Z
LT000000031,UC100000,MD500000,2025-02-06,2025-02,17373,17726,353,ESTIMADA,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv,2026-10-19T20:11:27.087Z
LT000000032,UC100000,MD500000,2025-03-25,2025-03,17726,18132,407,REAL,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv,2026-10-19T20:11:27.087Z
LT000000033,UC100000,MD500000,2025-04-22,2025-04,18132,18496,363,ESTIMADA,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv,2026-10-19T20:11:27.087Z
LT000000034,UC100000,MD500000,2025-05-10,2025-05,18496,18753,257,REAL,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv,2026-10-19T20:11:27.087Z


## 4. Cópias exatas de linhas (log L1)

**Decisão:** `dropDuplicates(["id_leitura"])`.

**Evidência:** a Bronze tem 2.182.468 linhas para 2.160.945 `id_leitura` distintos, ou seja, 21.523 repetições. Sem as colunas de auditoria (`source_file` e `ingestion_timestamp`), as linhas distintas também são 2.160.945. Logo, as repetições são idênticas, e manter qualquer uma delas não perde informação.

**Por que primeiro:** todas as contagens seguintes partem daqui.

**Conferência:** 21.523 linhas removidas; restam 2.160.945.

In [0]:
# Tratamento 1 · cópias exatas (esperado: 21.523 removidas)
def n_br(n):
    return f"{n:,}".replace(",", ".")

df_dedup = bronze.dropDuplicates(["id_leitura"])
n_dedup = df_dedup.count()
print(f"Cópias removidas: {n_br(n_bronze - n_dedup)} | antes: {n_br(n_bronze)} | depois: {n_br(n_dedup)}")

Cópias removidas: 21.523 | antes: 2.182.468 | depois: 2.160.945


## 5. Linhas órfãs: `id_medidor` inexistente no cadastro (log L2)

**Decisão:** remover.

**Alternativa considerada:** gravar `df_orfas` em uma tabela de quarentena. Descartada porque a Bronze já guarda as 8.609 linhas intactas (basta filtrar o `id_medidor` fora do padrão). Ressalva: a Bronze é sobrescrita a cada carga, então, se o pipeline rodar com dados novos, a quarentena passaria a ser o único histórico do que foi rejeitado.

**Evidência:** o id válido é `MD` + 6 dígitos. 8.609 linhas têm id com 7 dígitos, e nenhum deles existe em `medidores`. Cada uma está em par com uma linha válida da mesma UC e do mesmo mês, e é isso que gera pares `(id_uc, mes_referencia)` repetidos. Se fossem mantidas, o `id_medidor` da UC "pularia" para um id inválido e voltaria, criando trocas de medidor falsas: 23.314 antes da remoção, 6.299 depois. O case exige que nenhuma análise dependa de registros órfãos.

**Como:** join `left_semi` (mantém) e `left_anti` (órfãs) contra os ids **distintos** de `medidores`. Um join comum multiplicaria linhas, porque `medidores` tem 513 linhas com id repetido. Todos os `id_uc` das leituras restantes existem em `unidades_consumidoras` (verificado na exploração).

**Conferência:** 8.609 linhas removidas; restam 2.152.336, com `(id_uc, mes_referencia)` único.

In [0]:
# Tratamento 2 · órfãs: id_medidor inexistente no cadastro (esperado: 8.609 removidas, 2.152.336 restantes)
ids_medidores = spark.table(f"{bronze_schema}.medidores").select("id_medidor").distinct()  # distinct: medidores tem ids repetidos

df_orfas = df_dedup.join(ids_medidores, "id_medidor", "left_anti")
df_validas = df_dedup.join(ids_medidores, "id_medidor", "left_semi")

n_validas = df_validas.count()
print(f"Órfãs removidas: {n_br(n_dedup - n_validas)} | antes: {n_br(n_dedup)} | depois: {n_br(n_validas)}")
display(df_orfas.select("id_leitura", "id_uc", "id_medidor", "mes_referencia").limit(5))

Órfãs removidas: 8.609 | antes: 2.160.945 | depois: 2.152.336


id_leitura,id_uc,id_medidor,mes_referencia
LTX0000039,UC135617,MD9586715,2025-02
LTX0000076,UC126338,MD9293150,2025-05
LTX0000251,UC100758,MD9314830,2025-07
LTX0000552,UC126657,MD9662601,2025-07
LTX0000584,UC139663,MD9336997,2025-06


## 6. Marcas lidas do texto original, antes de qualquer cast (log L3 e L4)

**Decisão:** registrar duas marcas olhando o **texto** das colunas, antes de convertê-las, porque o cast apaga essa informação.

- **`leitura_anterior_ausente`:** a leitura anterior aparece "vazia" de quatro formas: nulo real (8.101), `N/A` (8.149), `NULL` (8.073) e o código `-1` (7.962). Total: 32.285 linhas (1,5%). O consumo está preenchido e numérico em todas elas, e só 339 dos 16.222 casos de `N/A`/`NULL` são a primeira leitura da UC. Ou seja, é dado que faltou, e não ausência com significado de negócio. A flag guarda essa informação depois que as quatro formas viram um único nulo.
- **`consumo_ponto_zero` (auxiliar):** consumo negativo escrito com `.0` no texto (ex.: `-85.0`) é sinal invertido. Todo consumo com `.0` na tabela é negativo, e nenhum consumo negativo por troca de medidor tem `.0`. Essa coluna é usada na seção 9 e não vai para a Silver.

**Conferência:** 32.285 linhas com a leitura anterior ausente (divididas nas quatro formas acima) e 1.717 com `.0`.

In [0]:
# Tratamentos 3 e 4 (parte 1) · marcas lidas do TEXTO original, antes de qualquer cast
df_marcas = (
    df_validas
    .withColumn("consumo_ponto_zero", F.col("consumo_kwh").endswith(".0"))  # auxiliar: marca o sinal invertido
    .withColumn(
        "leitura_anterior_ausente",
        F.col("leitura_anterior").isNull() | F.col("leitura_anterior").isin("N/A", "NULL", "-1"),
    )
)

forma = (
    F.when(F.col("leitura_anterior").isNull(), "nulo_real")
     .when(F.col("leitura_anterior") == "N/A", "texto_N/A")
     .when(F.col("leitura_anterior") == "NULL", "texto_NULL")
     .otherwise("codigo_-1")
)
# esperado: nulo_real 8.101 | texto_N/A 8.149 | texto_NULL 8.073 | codigo_-1 7.962 (total 32.285)
display(df_marcas.filter("leitura_anterior_ausente").groupBy(forma.alias("forma")).count().orderBy("forma"))

# esperado: 1.717 com ".0" (sinal invertido)
print("Consumo com '.0' no texto:", n_br(df_marcas.filter("consumo_ponto_zero").count()))

forma,count
codigo_-1,7962
nulo_real,8101
texto_N/A,8149
texto_NULL,8073


Consumo com '.0' no texto: 1.717


## 7. Conversão numérica (log L3 e L4)

**Decisões:**
- `try_cast` em todas as conversões. O `cast` comum dá erro com `N/A` e `NULL`.
- **`leitura_anterior` ausente vira nulo real e não é preenchida.** É um nulo que preservamos (o registro continua utilizável, porque o consumo existe), sem inventar um valor calculado. Alternativa considerada: a coluna `leitura_anterior_recuperada` (ver log L3), não incluída por ora. A flag da seção 6 mantém a informação.
- **O consumo passa por `double` antes do `bigint`:** valores como `-125.0` não convertem direto para inteiro. A coluna auxiliar `consumo_double` serve para uma checagem: se existisse algum consumo com casa decimal diferente de `.0`, o cast para inteiro truncaria o valor sem avisar.
- `consumo_kwh_original` guarda o consumo como veio da origem (já numérico), para auditoria depois que a coluna `consumo_kwh` for corrigida.

**Conferência:** nenhuma `leitura_atual` nem consumo nulo depois do cast.

In [0]:
# Tratamento 3 (parte 2) · conversão numérica com try_cast; as 4 formas de ausência viram nulo real
df_tipos = (
    df_marcas
    .withColumn("leitura_atual", F.expr("try_cast(leitura_atual as bigint)"))
    .withColumn(
        "leitura_anterior",
        F.when(F.col("leitura_anterior_ausente"), F.lit(None).cast("bigint"))
         .otherwise(F.expr("try_cast(leitura_anterior as bigint)")),
    )
    # consumo passa por double porque "-125.0" não converte direto para bigint
    .withColumn("consumo_double", F.expr("try_cast(consumo_kwh as double)"))  # auxiliar, usada em uma checagem
    .withColumn("consumo_kwh_original", F.expr("try_cast(consumo_double as bigint)"))
    .drop("consumo_kwh")  # a coluna final consumo_kwh é criada no tratamento 6
)

# esperado: 0 em ambos (a conversão não pode anular nenhum valor)
print("leitura_atual nula após o cast:", n_br(df_tipos.filter("leitura_atual IS NULL").count()))
print("consumo nulo após o cast:      ", n_br(df_tipos.filter("consumo_kwh_original IS NULL").count()))

leitura_atual nula após o cast: 0
consumo nulo após o cast:       0


## 8. `data_leitura` em quatro formatos (log L6)

**Decisão:** converter cada padrão com o seu formato e gravar como `date`.

| Padrão | Formato | Linhas |
|---|---|---|
| `9999-99-99` | `yyyy-MM-dd` | 1.894.056 |
| `99.99.9999` | `dd.MM.yyyy` | 86.433 |
| `99/99/9999` | `dd/MM/yyyy` | 85.936 |
| `99-99-9999` | `MM-dd-yyyy` (**mês primeiro**) | 85.911 |

**Evidência da ordem dia/mês:** comparando o mês de cada data convertida com `mes_referencia`, os quatro formatos acima convertem sem falhas e batem em 100% das linhas. Com `dd-MM-yyyy` no padrão com hífen, 49.082 das 85.911 linhas não convertiam, e só 3.132 batiam.

**Risco evitado:** aplicar um formato único converteria algumas datas para o mês errado, sem dar erro.

`mes_referencia` já está limpo (todas as linhas em `yyyy-MM`) e permanece como texto.

**Conferência:** nenhuma data nula depois da conversão e nenhum mês diferente de `mes_referencia`.

In [0]:
# Tratamento 6 · data_leitura em 4 formatos: cada padrão com o seu formato (hífen = MÊS primeiro)
formatos = {
    "9999-99-99": "yyyy-MM-dd",
    "99-99-9999": "MM-dd-yyyy",
    "99.99.9999": "dd.MM.yyyy",
    "99/99/9999": "dd/MM/yyyy",
}

data_convertida = None
for padrao, fmt in formatos.items():
    cond = F.col("padrao_data") == padrao
    valor = F.expr(f"to_date(try_to_timestamp(data_leitura, '{fmt}'))")
    data_convertida = F.when(cond, valor) if data_convertida is None else data_convertida.when(cond, valor)

df_datas = (
    df_tipos
    .withColumn("padrao_data", F.regexp_replace("data_leitura", "[0-9]", "9"))
    .withColumn("data_leitura", data_convertida)  # lê o texto original e substitui pela data
)

# esperado: 0 datas que não converteram
print("data_leitura nula após a conversão:", n_br(df_datas.filter("data_leitura IS NULL").count()))
# esperado: 100% das datas no mesmo mês de mes_referencia
print("mês da data diferente de mes_referencia:",
      n_br(df_datas.filter(F.date_format("data_leitura", "yyyy-MM") != F.col("mes_referencia")).count()))

data_leitura nula após a conversão: 0
mês da data diferente de mes_referencia: 0


## 9. Troca de medidor e consumo final (log L4, L5 e L7)

Todo consumo negativo tem uma de duas causas, e o critério do medidor separa as duas sem sobreposição:

**Troca de medidor (log L5), 6.299 linhas.** O `id_medidor` da UC é diferente do mês anterior. A `leitura_anterior` é a última do medidor antigo e a `leitura_atual` é a do medidor novo, então `atual − anterior` não faz sentido. Só ocorre de 2025-09 a 2026-03.
- **Decisão:** `consumo_kwh = leitura_atual` + flag `troca_medidor`.
- **Alternativa considerada:** consumo nulo + flag (muda uma linha do código). Descartada porque faria o volume dos meses recentes parecer menor do que foi e porque o cadastro e as faturas sustentam a leitura atual como consumo. O valor original continua em `consumo_kwh_original`.
- **Evidências:** no cadastro, o medidor antigo está `RETIRADO` com `data_retirada` no mesmo mês da troca em 100% dos casos, e o medidor novo pertence à mesma UC. Nas faturas, o preço implícito calculado com `leitura_atual` como consumo fica ~5% acima do preço típico da UC (mediana 1,05): evidência forte, mas não prova.
- **Impacto na Gold:** essas linhas caem nos últimos 12 meses, justamente os que o diretor compara com o primeiro ano. Consumo nulo faria o volume recente parecer menor do que foi.

**Sinal invertido (log L4), 1.717 linhas.** Leituras coerentes, medidor sem troca e consumo com o sinal trocado (escrito com `.0`).
- **Decisão:** `abs(consumo)` + flag `consumo_corrigido`.
- **Alternativas consideradas:** consumo nulo (perderia um consumo que existe e está confirmado) e recalcular `atual − anterior` (mudaria ~24% das linhas em 1 kWh, por causa do ±1).
- **Evidência:** nenhuma linha de sinal invertido tem troca de medidor, e toda regressão de leitura tem. O critério do `.0` e o do medidor dão a mesma classificação. Nas 30 linhas com `leitura_anterior` ausente, `abs(consumo)` bate com `atual − leitura anterior recuperada` (±1).

**Demais linhas (log #7).** O consumo original difere de `atual − anterior` em ±1 kWh em ~24% das linhas, de forma simétrica, o que parece arredondamento da origem. **Não corrigimos:** recalcular mudaria ~24% das linhas em 1 kWh. O ±1 vira tolerância na checagem de coerência (seção 11).

**Ordem:** a troca (`lag` por UC) só pode ser calculada depois de remover cópias e órfãs, senão as órfãs criam trocas falsas.

**Conferência:** 6.299 trocas, 1.717 consumos corrigidos, nenhuma linha nas duas categorias e nenhum consumo negativo sem explicação.

In [0]:
# Tratamentos 4 e 5 · troca de medidor e consumo final (só depois de remover cópias e órfãs)
w_uc = Window.partitionBy("id_uc").orderBy("mes_referencia")

df_consumo = (
    df_datas
    .withColumn("medidor_anterior", F.lag("id_medidor").over(w_uc))
    .withColumn("troca_medidor", F.col("medidor_anterior").isNotNull() & (F.col("id_medidor") != F.col("medidor_anterior")))
    .withColumn("consumo_corrigido", F.col("consumo_ponto_zero"))
    .withColumn(
        "consumo_kwh",
        F.when(F.col("troca_medidor"), F.col("leitura_atual"))                    # troca: leitura do medidor novo
         .when(F.col("consumo_corrigido"), F.abs(F.col("consumo_kwh_original")))  # sinal invertido
         .otherwise(F.col("consumo_kwh_original")),
    )
)

# esperado: troca 6.299 | sinal invertido 1.717 | sobreposição entre os dois: 0 | negativos sem explicação: 0
print("troca_medidor:     ", n_br(df_consumo.filter("troca_medidor").count()))
print("consumo_corrigido: ", n_br(df_consumo.filter("consumo_corrigido").count()))
print("troca E sinal invertido:", n_br(df_consumo.filter("troca_medidor AND consumo_corrigido").count()))
print("negativos sem explicação:",
      n_br(df_consumo.filter("consumo_kwh_original < 0 AND NOT troca_medidor AND NOT consumo_corrigido").count()))

troca_medidor:      6.299
consumo_corrigido:  1.717
troca E sinal invertido: 0
negativos sem explicação: 0


## 10. Seleção final das colunas

Colunas auxiliares (`consumo_ponto_zero`, `consumo_double`, `padrao_data`, `medidor_anterior`) ficam de fora. `tipo_leitura` não tinha variações de grafia; `upper(trim)` só garante a padronização. Ficam três flags para a Gold saber o que foi tratado em cada linha: `leitura_anterior_ausente`, `consumo_corrigido` e `troca_medidor`.

In [0]:
# Seleção final das colunas da Silver (descarta as colunas auxiliares)
df_silver = (
    df_consumo
    .withColumn("tipo_leitura", F.upper(F.trim("tipo_leitura")))
    .select(
        "id_leitura", "id_uc", "id_medidor", "mes_referencia", "data_leitura",
        "leitura_anterior", "leitura_atual", "consumo_kwh", "consumo_kwh_original", "tipo_leitura",
        "leitura_anterior_ausente", "consumo_corrigido", "troca_medidor",
        "source_file",
    )
)
df_silver.printSchema()

root
 |-- id_leitura: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- id_medidor: string (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- data_leitura: date (nullable = true)
 |-- leitura_anterior: long (nullable = true)
 |-- leitura_atual: long (nullable = true)
 |-- consumo_kwh: long (nullable = true)
 |-- consumo_kwh_original: long (nullable = true)
 |-- tipo_leitura: string (nullable = true)
 |-- leitura_anterior_ausente: boolean (nullable = true)
 |-- consumo_corrigido: boolean (nullable = true)
 |-- troca_medidor: boolean (nullable = true)
 |-- source_file: string (nullable = true)



## 11. Checagens de qualidade

Rodam sobre o resultado, **antes** de gravar. Cada uma protege uma decisão acima:

| Checagem | Protege |
|---|---|
| Total de linhas = 2.152.336 | cópias e órfãs removidas na quantidade esperada (vale para esta extração) |
| `id_leitura` e `(id_uc, mes_referencia)` únicos | cópias e órfãs |
| Colunas-chave não nulas | conversões de número e de data |
| `id_medidor` no padrão `MD` + 6 dígitos | órfãs |
| `tipo_leitura` só `REAL` ou `ESTIMADA` | domínio da categoria |
| `consumo_kwh > 0` e `leitura_atual >= 0` | sinal invertido e troca de medidor |
| Mês de `data_leitura` = `mes_referencia` | ordem dia/mês das datas |
| `leitura_anterior` nula só quando marcada como ausente | as quatro formas de ausência |
| Consumo coerente com as leituras (±1), exceto troca e leitura ausente | tolerância do arredondamento da origem |
| Consumo da origem sem casas decimais | o cast para inteiro não truncou nada |
| `troca_medidor` e `consumo_corrigido` só em linhas com consumo original negativo | as regras não sobrescrevem um consumo positivo correto |
| Nenhuma linha com `troca_medidor` e `consumo_corrigido` ao mesmo tempo | as duas causas são disjuntas |

A checagem de contagem total é uma proteção desta extração única: se a Silver for reexecutada com dados novos, o valor esperado precisa ser atualizado.

In [0]:
# Checagens de qualidade antes de gravar (condição = o que deve ser verdade em todas as linhas)
t = "silver.leituras"
dq_results.clear()  # se esta célula for reexecutada, não acumula resultados antigos

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=diferenca, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

# Volume e chaves
dq_check_contagem(t, "total de linhas = 2.152.336", df_silver, 2_152_336)
dq_check_unique(t, "id_leitura único", df_silver, ["id_leitura"])
dq_check_unique(t, "(id_uc, mes_referencia) único", df_silver, ["id_uc", "mes_referencia"])

# Nulos e domínios
for coluna in ["id_leitura", "id_uc", "id_medidor", "data_leitura", "leitura_atual", "consumo_kwh"]:
    dq_check(t, f"{coluna} não nulo", df_silver, F.col(coluna).isNotNull())

dq_check(t, "id_medidor no padrão MD + 6 dígitos", df_silver, F.col("id_medidor").rlike("^MD[0-9]{6}$"))
dq_check(t, "tipo_leitura só REAL ou ESTIMADA", df_silver,
         F.coalesce(F.col("tipo_leitura").isin("REAL", "ESTIMADA"), F.lit(False)))  # nulo também falha

# Valores
dq_check(t, "consumo_kwh > 0", df_silver, F.col("consumo_kwh") > 0)
dq_check(t, "leitura_atual >= 0", df_silver, F.col("leitura_atual") >= 0)
dq_check(t, "mês de data_leitura = mes_referencia", df_silver,
         F.date_format("data_leitura", "yyyy-MM") == F.col("mes_referencia"))
dq_check(t, "leitura_anterior nula só quando marcada como ausente", df_silver,
         F.col("leitura_anterior").isNotNull() | F.col("leitura_anterior_ausente"))
# tolerância de ±1: o consumo da origem difere de atual - anterior em ±1 kWh por arredondamento
dq_check(t, "consumo coerente com as leituras (±1), exceto troca e leitura ausente", df_silver,
         F.col("troca_medidor") | F.col("leitura_anterior_ausente")
         | (F.abs(F.col("consumo_kwh") - (F.col("leitura_atual") - F.col("leitura_anterior"))) <= 1))

# Proteção das regras de tratamento
dq_check(t, "consumo da origem sem casas decimais (o cast para inteiro não truncou)", df_consumo,
         F.col("consumo_double") == F.col("consumo_kwh_original").cast("double"))
dq_check(t, "troca_medidor só em consumo original negativo", df_silver,
         ~F.col("troca_medidor") | (F.col("consumo_kwh_original") < 0))
dq_check(t, "consumo_corrigido só em consumo original negativo", df_silver,
         ~F.col("consumo_corrigido") | (F.col("consumo_kwh_original") < 0))
dq_check(t, "troca_medidor e consumo_corrigido nunca juntos", df_silver,
         ~(F.col("troca_medidor") & F.col("consumo_corrigido")))

[PASS] silver.leituras | total de linhas = 2.152.336 | 2.152.336 linhas (esperado 2.152.336)
[PASS] silver.leituras | id_leitura único | 0 chaves duplicadas de 2152336 linhas
[PASS] silver.leituras | (id_uc, mes_referencia) único | 0 chaves duplicadas de 2152336 linhas
[PASS] silver.leituras | id_leitura não nulo | 0/2152336 linhas falharam
[PASS] silver.leituras | id_uc não nulo | 0/2152336 linhas falharam
[PASS] silver.leituras | id_medidor não nulo | 0/2152336 linhas falharam
[PASS] silver.leituras | data_leitura não nulo | 0/2152336 linhas falharam
[PASS] silver.leituras | leitura_atual não nulo | 0/2152336 linhas falharam
[PASS] silver.leituras | consumo_kwh não nulo | 0/2152336 linhas falharam
[PASS] silver.leituras | id_medidor no padrão MD + 6 dígitos | 0/2152336 linhas falharam
[PASS] silver.leituras | tipo_leitura só REAL ou ESTIMADA | 0/2152336 linhas falharam
[PASS] silver.leituras | consumo_kwh > 0 | 0/2152336 linhas falharam
[PASS] silver.leituras | leitura_atual >= 0 | 0

## 12. Registro das checagens em `silver.dq_log`

O resultado das checagens é gravado **sempre**, mesmo quando alguma falha: o registro de uma falha é útil para investigar.

A gravação usa o modo `append` (o mesmo da aula), então **cada execução acrescenta linhas** e a tabela acumula as checagens de todas as execuções. O que diferencia uma execução da outra é a coluna `checked_at`. Ao consultar o log, filtre por `table_name` e pela execução mais recente, senão as mesmas checagens aparecem repetidas. A célula abaixo mostra só a execução atual.

In [0]:
# Persistência do dq_log (modo append, igual à aula); sempre grava, inclusive quando há falha
df_dq_log = spark.createDataFrame(dq_results)
df_dq_log.write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{silver_schema}.dq_log")

# mostra só a execução atual (as últimas len(dq_results) linhas desta tabela)
display(
    spark.table(f"{silver_schema}.dq_log")
    .filter(F.col("table_name") == "silver.leituras")
    .orderBy(F.col("checked_at").desc())
    .limit(len(dq_results))
)

table_name,check_name,total_rows,failed_rows,passed,checked_at
silver.leituras,troca_medidor e consumo_corrigido nunca juntos,2152336,0,true,2026-10-20T20:45:28.254Z
silver.leituras,consumo_corrigido só em consumo original negativo,2152336,0,true,2026-10-20T20:45:25.532Z
silver.leituras,troca_medidor só em consumo original negativo,2152336,0,true,2026-10-20T20:45:23.540Z
silver.leituras,consumo da origem sem casas decimais (o cast para inteiro não truncou),2152336,0,true,2026-10-20T20:45:21.056Z
silver.leituras,"consumo coerente com as leituras (±1), exceto troca e leitura ausente",2152336,0,true,2026-10-20T20:45:19.165Z
silver.leituras,leitura_anterior nula só quando marcada como ausente,2152336,0,true,2026-10-20T20:45:16.723Z
silver.leituras,mês de data_leitura = mes_referencia,2152336,0,true,2026-10-20T20:45:14.659Z
silver.leituras,leitura_atual >= 0,2152336,0,true,2026-10-20T20:45:11.873Z
silver.leituras,consumo_kwh > 0,2152336,0,true,2026-10-20T20:45:09.969Z
silver.leituras,tipo_leitura só REAL ou ESTIMADA,2152336,0,true,2026-10-20T20:45:07.534Z


## 13. Gravação da Silver, só se todas as checagens passarem

A Gold lê a Silver. Por isso, se qualquer checagem falhar, a tabela **não é gravada**: o notebook imprime quais checagens falharam e é interrompido com erro (num Workflow, isso marca a execução como falha e impede que dados com problema sigam para as camadas seguintes). Se `silver.leituras` já existia de uma execução anterior, ela permanece como estava.

In [0]:
# Gravação da Silver: só acontece se todas as checagens passaram
falhas = [r for r in dq_results if not r.passed]

if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_silver} NÃO foi gravada:")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_silver} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

(
    df_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_silver)
)
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_silver} gravada: "
      f"{n_br(spark.table(tabela_silver).count())} linhas")

Todas as 20 checagens passaram. v_energy.silver.leituras gravada: 2.152.336 linhas


## 14. Bronze × Silver, com as mesmas linhas

Para cada tratamento, os `id_leitura` são escolhidos e depois mostrados nas duas tabelas, então dá para ver o **texto original** da Bronze (`-85.0`, `N/A`, `10/09/2022`) ao lado do valor tratado na Silver. Nos casos de cópia e órfã, a Bronze mostra as linhas e a Silver mostra o resultado da remoção (uma linha no primeiro caso e nenhuma no segundo).

In [0]:
# Bronze × Silver: os MESMOS id_leitura nas duas tabelas, um grupo de exemplos por tratamento
silver_t = spark.table(tabela_silver)
ids_silver = silver_t.select("id_leitura")
colunas_bronze = ["id_leitura", "id_uc", "id_medidor", "mes_referencia", "data_leitura",
                  "leitura_anterior", "leitura_atual", "consumo_kwh", "tipo_leitura"]

def comparar(titulo, ids_df, n=3):
    ids = [r["id_leitura"] for r in ids_df.select("id_leitura").limit(n).collect()]
    print(f"=== {titulo} | ids: {ids}")
    print("Bronze (texto original):")
    display(bronze.filter(F.col("id_leitura").isin(ids)).select(*colunas_bronze).orderBy("id_leitura"))
    print("Silver (tratado):")
    display(silver_t.filter(F.col("id_leitura").isin(ids)).orderBy("id_leitura"))

# 1. Cópias: 2 linhas idênticas na Bronze, 1 na Silver
comparar("Cópias exatas", bronze.groupBy("id_leitura").count().filter("count > 1"), n=2)

# 2. Órfãs: existem na Bronze, não existem na Silver
comparar("Órfãs (id_medidor inexistente)", df_orfas, n=2)

# 3. leitura_anterior ausente: um exemplo de cada forma (nulo real, N/A, NULL, -1)
ids_ausente = (
    bronze.join(ids_silver, "id_leitura", "left_semi")
    .filter(F.col("leitura_anterior").isNull() | F.col("leitura_anterior").isin("N/A", "NULL", "-1"))
    .withColumn("forma", forma)
    .dropDuplicates(["forma"])
)
comparar("leitura_anterior ausente (uma de cada forma)", ids_ausente, n=4)

# 4. Sinal invertido: consumo negativo com .0 vira positivo
comparar("Sinal invertido", silver_t.filter("consumo_corrigido"))

# 5. Troca de medidor: consumo negativo vira leitura_atual
comparar("Troca de medidor", silver_t.filter("troca_medidor"))

# 6. Datas: um exemplo de cada formato fora do ISO
ids_datas = (
    bronze.join(ids_silver, "id_leitura", "left_semi")
    .withColumn("padrao", F.regexp_replace("data_leitura", "[0-9]", "9"))
    .filter("padrao != '9999-99-99'")
    .dropDuplicates(["padrao"])
)
comparar("data_leitura fora do ISO (um de cada formato)", ids_datas, n=3)

=== Cópias exatas | ids: ['LT000005692', 'LT000058075']
Bronze (texto original):


id_leitura,id_uc,id_medidor,mes_referencia,data_leitura,leitura_anterior,leitura_atual,consumo_kwh,tipo_leitura
LT000005692,UC100118,MD500118,2025-05,2025-05-26,80955,81284,328,ESTIMADA
LT000005692,UC100118,MD500118,2025-05,2025-05-26,80955,81284,328,ESTIMADA
LT000058075,UC101212,MD501212,2025-02,2025-02-27,87322,87725,403,REAL
LT000058075,UC101212,MD501212,2025-02,2025-02-27,87322,87725,403,REAL


Silver (tratado):


id_leitura,id_uc,id_medidor,mes_referencia,data_leitura,leitura_anterior,leitura_atual,consumo_kwh,consumo_kwh_original,tipo_leitura,leitura_anterior_ausente,consumo_corrigido,troca_medidor,source_file
LT000005692,UC100118,MD500118,2025-05,2025-05-26,80955,81284,328,328,ESTIMADA,false,false,false,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv
LT000058075,UC101212,MD501212,2025-02,2025-02-27,87322,87725,403,403,REAL,false,false,false,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv


=== Órfãs (id_medidor inexistente) | ids: ['LTX0000039', 'LTX0000076']
Bronze (texto original):


...

id_leitura,id_uc,id_medidor,mes_referencia,data_leitura,leitura_anterior,leitura_atual,consumo_kwh,consumo_kwh_original,tipo_leitura,leitura_anterior_ausente,consumo_corrigido,troca_medidor,source_file
LT000000569,UC100011,MD545218,2025-12,2025-12-27,195613,4698,4698,-190915,REAL,false,false,true,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv
LT000000759,UC100015,MD545821,2025-10,2025-10-24,79980,80,80,-79900,REAL,false,false,true,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv
LT000001145,UC100023,MD549795,2025-12,2025-12-24,55647,250,250,-55397,REAL,false,false,true,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv


=== data_leitura fora do ISO (um de cada formato) | ids: ['LT000000177', 'LT000000081', 'LT000000135']
Bronze (texto original):


id_leitura,id_uc,id_medidor,mes_referencia,data_leitura,leitura_anterior,leitura_atual,consumo_kwh,tipo_leitura
LT000000081,UC100001,MD500001,2025-04,04-13-2025,45462,46492,1031,ESTIMADA
LT000000135,UC100002,MD500002,2025-10,12/10/2025,90447,90631,184,ESTIMADA
LT000000177,UC100003,MD500003,2025-04,17.04.2025,89018,89241,223,REAL
LT000000177,UC100003,MD500003,2025-04,17.04.2025,89018,89241,223,REAL


Silver (tratado):


id_leitura,id_uc,id_medidor,mes_referencia,data_leitura,leitura_anterior,leitura_atual,consumo_kwh,consumo_kwh_original,tipo_leitura,leitura_anterior_ausente,consumo_corrigido,troca_medidor,source_file
LT000000081,UC100001,MD500001,2025-04,2025-04-13,45462,46492,1031,1031,ESTIMADA,false,false,false,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv
LT000000135,UC100002,MD500002,2025-10,2025-10-12,90447,90631,184,184,ESTIMADA,false,false,false,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv
LT000000177,UC100003,MD500003,2025-04,2025-04-17,89018,89241,223,223,REAL,false,false,false,dbfs:/Volumes/v_energy/bronze/landing/leituras/leituras_2025.csv
